This notebook does:
- load airborne hyperspectral data from `data/you-shall-not-pass/Obrazy lotniczne`
- display false-color composite from the cube
- compute water quality metrics (Chl-a, DOC, turbidity) from airborne data
- stream Sentinel-2 patch close to airborne acquisition date from Microsoft Planetary Computer
- compute same indices on Sentinel-2 and compare

In [6]:
!uv add spectral pystac-client planetary-computer rasterio

Resolved 146 packages in 2ms
Audited 140 packages in 12ms


In [36]:
import os
from pathlib import Path
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import rasterio

BASE_DIR = Path(os.getcwd()) / 'data' / 'you-shall-not-pass' / 'mggp_naloty' / 'obrazy_lotnicze'

AQUISITION_DATE = '2025-07-17'
print('base path:', BASE_DIR)
print('path exists:', BASE_DIR.exists())

base path: E:\github_repos\eolabs\data\you-shall-not-pass\mggp_naloty\obrazy_lotnicze
path exists: True


In [37]:
hdr_files = glob(os.path.join(BASE_DIR, '**', '*.hdr'), recursive=True)
bsq_files = glob(os.path.join(BASE_DIR, '**', '*.bsq'), recursive=True)

print('hdr:', hdr_files)
print('bsq:', bsq_files)

hdr: ['E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_008_VS_join_atm.hdr', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_013_VS_join_atm.hdr', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_015_VS_join_atm.hdr']
bsq: ['E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_006_VS_join_atm.bsq', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_007_VS_join_atm.bsq', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_008_VS_join_atm.bsq', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_Blok_A_013_VS_join_atm.bsq', 'E:\\github_repos\\eolabs\\data\\you-shall-not-pass\\mggp_naloty\\obrazy_lotnicze\\221000_Odra_HS_B

In [38]:
import spectral.io.envi as envi

if hdr_files:
    hdr_path = hdr_files[0]
    img = envi.open(hdr_path)
    print("Opened:", hdr_path)
    print("Shape:", img.shape)
    
    meta = img.metadata
    wavelengths = None
    if 'wavelength' in meta:
        wavelengths = np.array([float(x) for x in meta['wavelength']])
    
    # We load the whole cube into memory
    cube = img.load()
else:
    raise FileNotFoundError('No hyperspectral files found')

Opened: E:\github_repos\eolabs\data\you-shall-not-pass\mggp_naloty\obrazy_lotnicze\221000_Odra_HS_Blok_A_008_VS_join_atm.hdr
Shape: (4300, 2001, 456)


MemoryError: Unable to allocate 14.6 GiB for an array with shape (4300, 2001, 456) and data type float32

In [39]:
samples = img.ncols
lines = img.nrows
bands = img.nbands
dtype = cube.dtype
interleave = 'bsq'
print(samples, lines, bands, interleave, dtype, wavelengths[:5] if wavelengths is not None else None)

NameError: name 'cube' is not defined

In [ ]:
# Cube is already loaded by img.load()
print('cube shape', cube.shape)

In [ ]:
# Display RGB False Color
from spectral import imshow

r_idx, g_idx, b_idx = 30, 20, 10
if wavelengths is not None:
    r_idx = np.argmin(np.abs(wavelengths - 670))
    g_idx = np.argmin(np.abs(wavelengths - 560))
    b_idx = np.argmin(np.abs(wavelengths - 490))

imshow(img, bands=(r_idx, g_idx, b_idx), title='Airborne False Color RGB')
plt.show()

In [ ]:
# Calculate water quality indices
def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

if wavelengths is not None:
    chl_band = np.argmin(np.abs(wavelengths - 705))
    ref_band = np.argmin(np.abs(wavelengths - 665))
    doc_band1 = np.argmin(np.abs(wavelengths - 560))
    turbidity_band = np.argmin(np.abs(wavelengths - 700))
else:
    chl_band, ref_band, doc_band1, turbidity_band = 85, 80, 45, 75

chla = safe_ratio(cube[:, :, chl_band], cube[:, :, ref_band])
doc = safe_ratio(cube[:, :, doc_band1], cube[:, :, ref_band])
turbidity = cube[:, :, turbidity_band]

print('chla mean:', np.nanmean(chla), 'doc mean:', np.nanmean(doc), 'turbidity mean:', np.nanmean(turbidity))

for label, data in [('Chl-a', chla), ('DOC', doc), ('Turbidity', turbidity)]:
    plt.figure(figsize=(6, 4))
    plt.imshow(data, cmap='viridis')
    plt.colorbar()
    plt.title(label)
    plt.show()

In [ ]:
x = samples // 2
y = lines // 2
print('sample index', x, 'line index', y)

spec = cube[y, x, :]
plt.figure(figsize=(8, 3))
plt.plot(wavelengths if wavelengths is not None else np.arange(bands), spec)
plt.title('Spectral signature at central pixel')
plt.xlabel('wavelength (nm)' if wavelengths is not None else 'band')
plt.ylabel('reflectance')
plt.grid()
plt.show()

In [ ]:
from pystac_client import Client
import planetary_computer
from datetime import datetime, timedelta

# Approximate Odra river bounding box, change as needed for exact flight ROI
bbox = [14.0, 53.0, 15.0, 54.0]

try:
    acq_date = datetime.strptime(AQUISITION_DATE, '%Y-%m-%d')
except ValueError:
    acq_date = datetime.strptime('2024-06-01', '%Y-%m-%d')

time_range = f"{(acq_date - timedelta(days=7)).strftime('%Y-%m-%d')}/{(acq_date + timedelta(days=7)).strftime('%Y-%m-%d')}"

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 30}}
)

items = list(search.items())
print('Found', len(items), 'Sentinel-2 scenes')

best_item = None
if items:
    best_item = items[0]
    print('Selected scene:', best_item.id)
    
    href_b4 = best_item.assets["B04"].href  # Red (665 nm)
    href_b3 = best_item.assets["B03"].href  # Green (560 nm)
    href_b8 = best_item.assets["B08"].href  # NIR (842 nm)
    href_b11 = best_item.assets["B11"].href # SWIR (1610 nm)
    
    print('Assets ready for streaming via Rasterio.')
else:
    print('No products found; please adjust bbox/date')

In [ ]:
if best_item:
    from rasterio.windows import Window
    
    # Read a 1000x1000 patch from the center or arbitrary window
    window = Window(1000, 1000, 1000, 1000) 
    
    band_data = {}
    with rasterio.open(href_b4) as src:
        band_data['B04'] = src.read(1, window=window).astype('float32') / 10000.0
    with rasterio.open(href_b3) as src:
        band_data['B03'] = src.read(1, window=window).astype('float32') / 10000.0
    with rasterio.open(href_b8) as src:
        band_data['B08'] = src.read(1, window=window).astype('float32') / 10000.0
    # B11 is 20m res, so read 500x500 and resize
    with rasterio.open(href_b11) as src:
        b11_raw = src.read(1, window=Window(500, 500, 500, 500)).astype('float32') / 10000.0
        # upsample to match 10m
        band_data['B11'] = b11_raw.repeat(2, axis=0).repeat(2, axis=1)

    chl_s2 = safe_ratio(band_data['B08'], band_data['B04'])
    doc_s2 = safe_ratio(band_data['B03'], band_data['B04'])
    turb_s2 = safe_ratio(band_data['B11'], band_data['B08'])

    for label, data in [('S2 Chl-a', chl_s2), ('S2 DOC', doc_s2), ('S2 Turbidity', turb_s2)]:
        plt.figure(figsize=(6, 4))
        plt.imshow(data, cmap='viridis')
        plt.colorbar()
        plt.title(label)
        plt.show()
else:
    chl_s2, doc_s2, turb_s2 = None, None, None

In [ ]:
if best_item and chl_s2 is not None:
    air_x, air_y = x, y
    s2_x, s2_y = chl_s2.shape[1] // 2, chl_s2.shape[0] // 2

    print('airborne indices at center:',
          'chla', chla[air_y, air_x],
          'doc', doc[air_y, air_x],
          'turb', turbidity[air_y, air_x])
    print('sentinel2 indices at center:',
          'chla', chl_s2[s2_y, s2_x],
          'doc', doc_s2[s2_y, s2_x],
          'turb', turb_s2[s2_y, s2_x])

    # scatter comparison at coarse sample
    N = min(chla.size, chl_s2.size)
    air_idx_vals = np.column_stack((chla.flatten()[:N], doc.flatten()[:N], turbidity.flatten()[:N]))
    
    # remove nans for plot
    valid = ~np.isnan(air_idx_vals).any(axis=1)
    
    s2_idx_vals = np.column_stack((chl_s2.flatten()[:N], doc_s2.flatten()[:N], turb_s2.flatten()[:N]))

    plt.figure(figsize=(10, 3))
    for i, name in enumerate(['Chl-a', 'DOC', 'Turbidity']):
        plt.subplot(1, 3, i + 1)
        plt.scatter(air_idx_vals[valid, i], s2_idx_vals[valid, i], s=1, alpha=0.5)
        plt.xlabel('Airborne')
        plt.ylabel('Sentinel-2')
        plt.title(name)
    plt.tight_layout()
    plt.show()

- set correct ROI footprint and acquisition date.
- for accurate water quality indices use published formulas and validation data.
- if `sentinelsat` is not installed, run `pip install sentinelsat`.